In [ ]:
# ========== 第 8 周练习：导入、鉴权、加载商品数据集 ==========
# 理念：向量库 + RAG + Modal 专家模型做合奏估价；本格先准备依赖与数据
# os：读环境变量（如 HF_TOKEN）
import os
# logging：后面代理用彩色前缀打日志
import logging
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# login：Hugging Face Hub 登录（拉数据集 / 模型）
from huggingface_hub import login
# re：从模型回复里用正则抠价格数字
import re
# SentenceTransformer：把商品摘要编码成向量（embedding）
from sentence_transformers import SentenceTransformer
# chromadb：本地持久化向量库，做相似商品检索
import chromadb
# TSNE：本格导入保留（后续可视化可能用）；逻辑不改
from sklearn.manifold import TSNE
# litellm.completion：统一接口调用多家 LLM
from litellm import completion
# tqdm.notebook：在 Jupyter 里显示进度条
from tqdm.notebook import tqdm
# Item：同目录 items.py 的数据类，支持 from_hub
from items import Item

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)

# Chroma 持久化目录名（相对当前工作目录）
DB = "products_vectorstore"

# 从环境读取 Hugging Face token（需在 .env 配置 HF_TOKEN）
hf_token = os.environ['HF_TOKEN']
# 登录 HF；不把凭证写进 git
login(token=hf_token, add_to_git_credential=False)

# 课程数据集命名空间
username = "ed-donner"
# 拼出 lite 版 items 数据集 ID
dataset = f"{username}/items_lite"

# 一次拉 train / val / test 三份 Item 列表
train, val, test = Item.from_hub(dataset)

# 打印规模，确认加载成功
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


In [ ]:
# ========== Chroma：打开（或创建）商品向量集合 ==========
# PersistentClient：向量与元数据落盘到 DB 路径
client = chromadb.PersistentClient(path=DB)
# 集合名 products（需事先写入 embedding，否则 query 为空）
collection_name = "products"
# get_or_create：有则取、无则建
collection = client.get_or_create_collection(collection_name)


In [ ]:
# ========== RAG 辅助 + Modal 专家价 + GPT RAG ==========
# 轻量句向量模型：与建库时尽量同一 encoder
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 从模型回复文本中提取第一个价格数字
def get_price(reply):
    # 去掉 $ 与千分位逗号
    reply = reply.replace("$", "").replace(",", "")
    # 正则匹配整数或小数
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    # 匹配成功则转 float，否则 0
    return float(match.group()) if match else 0

# 对 Item.summary 做句向量，供 Chroma query
def vector(item):
    # encoder.encode 返回 numpy 向量
    return encoder.encode(item.summary)

# 把相似商品+价格拼进 prompt 上下文（英文 prompt 保持原样）
def make_context(similars, prices):
    # 上下文引导语：告诉模型这些是参考商品
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    # 逐条追加相似品描述与价格
    for similar, price in zip(similars, prices):
        # 格式：产品文本 + Price is $xx
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    # 返回拼好的上下文字符串
    return message

# 组装 Chat messages；注意原文用了 summary 变量名
def messages_for(item, similars, prices):
    # 用户指令：只回价格、不要解释
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{summary}\n\n"
    # 追加相似品上下文
    message += make_context(similars, prices)
    # 单条 user message 列表
    return [{"role": "user", "content": message}]

# 编码 → Chroma 近邻 top-5 → 取出文档与 price 元数据
def find_similars(item):
    # 把 item 变成查询向量
    vec = vector(item)
    # query_embeddings 要 list[float]
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    # 取第一条 query 的 documents
    documents = results['documents'][0][:]
    # 从 metadatas 里抽 price 字段
    prices = [m['price'] for m in results['metadatas'][0][:]]
    # 返回 (文档列表, 价格列表)
    return documents, prices

# Modal：远程调用已部署的 pricer-service
import modal
# 按名称拿到远程 Pricer 类
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
# 实例化远程类句柄
pricer = Pricer() 

# 专家模型：摘要 → Modal price.remote
def specialist(summary):
    # 远程推理返回估价
    return pricer.price.remote(summary)

# Frontier RAG：相似品检索 + gpt-5.1 估价
def gpt_5__1_rag(item):
    # 先找相似品
    documents, prices = find_similars(item)
    # litellm completion；model id / seed 保持原样
    response = completion(model="openai/gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    # 返回模型文本内容（后续可再 get_price）
    return response.choices[0].message.content

# 合奏加权草稿（注释思路；真正合奏在 EnsembleAgent）
# 定义合奏（项目）：
# 价格1 = get_price(gpt_5__1_rag(项目))
# 价格2 = 专家(项目)
# # 价格3 = deep_neural_network(item)
# 返回价格1 * 0.8 + 价格2 * 0.2


In [ ]:
# ========== 日志：根 logger 设为 INFO，让 Agent.log 可见 ==========
# 拿到根 logger
root = logging.getLogger()
# INFO 及以上才会输出
root.setLevel(logging.INFO)


In [ ]:
# ========== Pydantic 业务模型 + test_scan 假扫描数据 ==========
# Deal / DealSelection / Opportunity 与课程管线一致
# Optional / List 类型标注
from typing import Optional, List
# BaseModel + Field：结构化字段与 schema 描述
from pydantic import BaseModel, Field

# Deal：一条交易
class Deal(BaseModel):
    """
    A class to Represent a Deal with a summary description
    """

    # Field.description 会进结构化输出 schema：英文必须保留
    product_description: str = Field(
        description="Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose."
    )
    # 广告/成交价字段（英文 description 保留）
    price: float = Field(
        description="The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200"
    )
    # 交易 URL 字段（英文 description 保留）
    url: str = Field(description="The URL of the deal, as provided in the input")


# DealSelection：多条 Deal 的容器
class DealSelection(BaseModel):
    """
    A class to Represent a list of Deals
    """

    # 要求模型挑描述清晰、价格明确的交易（英文 schema 保留）
    deals: List[Deal] = Field(
        description="Your selection of the 5 deals that have the most detailed, high quality description and the most clear price. You should be confident that the price reflects the deal, that it is a good deal, with a clear description"
    )

# Opportunity：交易 + 估价 + 折扣
class Opportunity(BaseModel):
    """
    A class to represent a possible opportunity: a Deal where we estimate
    it should cost more than it's being offered
    """

    # 原始交易
    deal: Deal
    # 模型估计公允价
    estimate: float
    # 折扣 = estimate - price
    discount: float
    
# 测试用扫描：不打真实 RSS，返回固定样例
def test_scan(memory: List[str] = []) -> Optional[DealSelection]:
        """
        Return a test DealSelection, to be used during testing
        """
        # results 字典结构对齐 DealSelection
        results = {
            "deals": [
                {
                    "product_description": "The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.",
                    "price": 178,
                    "url": "https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142",
                },
                {
                    "product_description": "The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.",
                    "price": 30,
                    "url": "https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39",
                },
                {
                    "product_description": "The Lenovo IdeaPad Slim 5 laptop is powered by a 7th generation AMD Ryzen 5 8645HS 6-core CPU, offering efficient performance for multitasking and demanding applications. It features a 16-inch touch display with a resolution of 1920x1080, ensuring bright and vivid visuals. Accompanied by 16GB of RAM and a 512GB SSD, the laptop provides ample speed and storage for all your files. This model is designed to handle everyday tasks with ease while delivering an enjoyable user experience.",
                    "price": 446,
                    "url": "https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-5-7-th-Gen-Ryzen-5-16-Touch-Laptop/485068.html?iref=rss-c39",
                },
                {
                    "product_description": "The Dell G15 gaming laptop is equipped with a 6th-generation AMD Ryzen 5 7640HS 6-Core CPU, providing powerful performance for gaming and content creation. It features a 15.6-inch 1080p display with a 120Hz refresh rate, allowing for smooth and responsive gameplay. With 16GB of RAM and a substantial 1TB NVMe M.2 SSD, this laptop ensures speedy performance and plenty of storage for games and applications. Additionally, it includes the Nvidia GeForce RTX 3050 GPU for enhanced graphics and gaming experiences.",
                    "price": 650,
                    "url": "https://www.dealnews.com/products/Dell/Dell-G15-Ryzen-5-15-6-Gaming-Laptop-w-Nvidia-RTX-3050/485067.html?iref=rss-c39",
                },
            ]
        }
        # 用 ** 解包构造成 DealSelection
        return DealSelection(**results)


In [ ]:
# logging：Agent.log 最终走到 logging.info
import logging

# Agent：所有代理的抽象基类（彩色日志前缀）
class Agent:
    """
    An abstract superclass for Agents
    Used to log messages in a way that can identify each Agent
    """

    # ANSI 前景色常量（终端着色）
    # 前景色
    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    MAGENTA = '\033[35m'
    CYAN = '\033[36m'
    WHITE = '\033[37m'
    
    # ANSI 背景色：黑底衬托彩色字
    # 背景颜色
    BG_BLACK = '\033[40m'
    
    # RESET：恢复终端默认颜色
    # 重置代码以返回默认颜色
    RESET = '\033[0m'

    # 子类覆盖：代理显示名
    name: str = ""
    # 子类覆盖：该代理的前景色
    color: str = '\033[37m'

    # 带颜色地打一条 INFO 日志，标明是哪个 Agent
    def log(self, message):
        """
        Log this as an info message, identifying the agent
        """
        # 背景黑 + 前景色
        color_code = self.BG_BLACK + self.color
        # 消息前加 [name]
        message = f"[{self.name}] {message}"
        # 输出后 RESET，避免污染后续终端颜色
        logging.info(color_code + message + self.RESET)


In [ ]:
# ========== MessagingAgent：Pushover 推送 + Claude 润色文案 ==========
# os：读 PUSHOVER_* 环境变量
import os
# litellm.completion：调用 Claude 写推送文案
from litellm import completion
# requests：POST 到 Pushover API
import requests

# Pushover 发送接口 URL（保持原样）
pushover_url = "https://api.pushover.net/1/messages.json"


# 继承 Agent：统一 log 风格
class MessagingAgent(Agent):
    # 代理显示名
    name = "Messaging Agent"
    # 日志颜色：白
    color = Agent.WHITE
    # 润色文案用的模型 id（保持原样）
    MODEL = "anthropic/claude-sonnet-4-5"

    # 初始化：读 Pushover 凭证
    def __init__(self):
        """
        Set up this object to either do push notifications via Pushover,
        or SMS via Twilio,
        whichever is specified in the constants
        """
        # 开始初始化日志
        self.log("Messaging Agent is initializing")
        # 用户 key；无环境变量时用占位默认值
        self.pushover_user = os.getenv("PUSHOVER_USER", "your-pushover-user-if-not-using-env")
        # 应用 token；无环境变量时用占位默认值
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "your-pushover-user-if-not-using-env")
        # 初始化完成日志
        self.log("Messaging Agent has initialized Pushover and Claude")

    # 调用 Pushover HTTP API 发推送
    def push(self, text):
        """
        Send a Push Notification using the Pushover API
        """
        # 准备发送日志
        self.log("Messaging Agent is sending a push notification")
        # 表单字段：user/token/message/sound
        payload = {
            "user": self.pushover_user,
            "token": self.pushover_token,
            "message": text,
            "sound": "cashregister",
        }
        # POST；data= 表单编码
        requests.post(pushover_url, data=payload)

    # 针对 Opportunity 拼简短告警并推送
    def alert(self, opportunity: Opportunity):
        """
        Make an alert about the specified Opportunity
        """
        # 拼价格 / 估价 / 折扣
        text = f"Deal Alert! Price=${opportunity.deal.price:.2f}, "
        text += f"Estimate=${opportunity.estimate:.2f}, "
        text += f"Discount=${opportunity.discount:.2f} :"
        # 描述前 10 字符 …
        text += opportunity.deal.product_description[:10] + "... "
        # 追加交易 URL
        text += opportunity.deal.url
        # 真正推送
        self.push(text)
        # 完成日志
        self.log("Messaging Agent has completed")

    # 用 Claude 把交易写成 2–3 句兴奋推送文案
    def craft_message(
        self, description: str, deal_price: float, estimated_true_value: float
    ) -> str:
        # user prompt 保持英文（影响模型输出）
        user_prompt = "Please summarize this great deal in 2-3 sentences to be sent as an exciting push notification alerting the user about this deal.\n"
        user_prompt += f"Item Description: {description}\nOffered Price: {deal_price}\nEstimated true value: {estimated_true_value}"
        user_prompt += "\n\nRespond only with the 2-3 sentence message which will be used to alert & excite the user about this deal"
        # litellm 调用；只传 user message
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        # 返回模型生成的文案
        return response.choices[0].message.content

    # 高层 notify：先润色再推送（截断 + URL）
    def notify(self, description: str, deal_price: float, estimated_true_value: float, url: str):
        """
        Make an alert about the specified details
        """
        # 开始用 Claude 写文案
        self.log("Messaging Agent is using Claude to craft the message")
        # 生成文案
        text = self.craft_message(description, deal_price, estimated_true_value)
        # 推送前 200 字 + URL，避免过长
        self.push(text[:200] + "... " + url)
        # 完成日志
        self.log("Messaging Agent has completed")


In [ ]:
# ========== EnsembleAgent：预处理改写 + 专家/Frontier 加权 ==========
# SYSTEM_PROMPT：把杂乱描述改写成固定字段（英文指令保留）
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

# 预处理用的 Groq/OSS 模型 id（保持原样）
PREPROCESSOR_MODEL = "groq/openai/gpt-oss-20b"

# 合奏代理：多模型估价后加权
class EnsembleAgent(Agent):
    # 显示名
    name = "Ensemble Agent"
    # 日志黄色
    color = Agent.YELLOW

    # 挂上 specialist 与 frontier 两个估价函数
    def __init__(self):
        """
        Create an instance of Ensemble, by creating each of the models
        And loading the weights of the Ensemble
        """
        # 初始化日志
        self.log("Initializing Ensemble Agent")
        # Modal 专家价函数（上一格定义）
        self.specialist = specialist
        # GPT RAG 估价函数（上一格定义）
        self.frontier = gpt_5__1_rag
        # 就绪日志
        self.log("Ensemble Agent is ready")
    
    # 组装预处理用的 system+user messages
    def messages_for(self, text: str) -> list[dict]:
        # system 放改写格式，user 放原始描述
        return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]

    # 调用预处理模型，得到结构化短描述
    def preprocess(self, text: str) -> str:
        # 带 SYSTEM_PROMPT 的 messages
        messages = self.messages_for(text)
        # completion；reasoning_effort / api_base 保持原参数
        response = completion(
            messages=messages,
            model=PREPROCESSOR_MODEL,
            reasoning_effort='low',
            api_base=None,
        )
        # 返回改写后的文本
        return response.choices[0].message.content

    # 合奏入口：预处理 → 两路估价 → 0.8/0.2 加权
    def price(self, description: str) -> float:
        """
        Run this ensemble model
        Ask each of the models to price the product
        Then use the Linear Regression model to return the weighted price
        :param description: the description of a product
        :return: an estimate of its price
        """
        # 开始预处理
        self.log("Running Ensemble Agent - preprocessing text")
        # 改写描述，便于下游模型
        rewrite = self.preprocess(description)
        # 记录用了哪个预处理模型
        self.log(f"Pre-processed text using {PREPROCESSOR_MODEL}")
        # 专家路径
        specialist = self.specialist(rewrite)
        # Frontier RAG 路径
        frontier = self.frontier(rewrite)
        # 线性加权合成最终估价
        combined = frontier * 0.8 + specialist * 0.2
        # 完成日志（含美元金额）
        self.log(f"Ensemble Agent complete - returning ${combined:.2f}")
        # 返回 combined
        return combined
    


In [ ]:
# ========== Memory：从 memory.json 恢复历史 Opportunity ==========
# json：读写记忆文件
import json
# 折扣阈值常量（后面 main 也会再赋一次）
DEAL_THRESHOLD = 50
# 记忆文件名（同目录 memory.json）
MEMORY_FILENAME = "memory.json"
# 若文件存在则反序列化为 Opportunity 列表
def read_memory() -> List[Opportunity]:
    # 文件存在才读
    if os.path.exists(MEMORY_FILENAME):
        # 打开 JSON
        with open(MEMORY_FILENAME, "r") as file:
            # load 成 list[dict]
            data = json.load(file)
        # dict → Pydantic Opportunity
        opportunities = [Opportunity(**item) for item in data]
        # 返回历史机会
        return opportunities
    # 无文件 → 空记忆
    return []

# 读入全局 memory，供本轮规划使用
memory = read_memory()


In [ ]:
# ========== Planning 主流程：扫描 → 合奏估价 → 阈值告警 ==========
# 消息代理实例
messenger = MessagingAgent()
# 合奏估价代理实例
ensemble = EnsembleAgent()

# 折扣阈值：$50
DEAL_THRESHOLD = 50

# 跑一轮：测试扫描 + 逐条估价 + 选最优
def main() -> Opportunity:
    # 用 test_scan 代替真实 RSS（memory 传入以保持签名）
    selection = test_scan(memory=memory)

    # 收集本轮 Opportunity
    opportunities = []

    # 逐条 Deal 估价
    for deal in selection.deals:
        # 规划日志：开始估价
        logging.info("Planning Agent is pricing up a potential deal")
        # 合奏代理返回 estimate
        estimate = ensemble.price(deal.product_description)
        # 折扣 = 估价 - 标价
        discount = estimate - deal.price
        # 记录该条折扣
        logging.info(f"Planning Agent has processed a deal with discount ${discount:.2f}")
        # 累加到 opportunities（保持原写法）
        opportunities += Opportunity(deal=deal, estimate=estimate, discount=discount)
    
    # 按折扣降序
    opportunities.sort(key=lambda opp: opp.discount, reverse=True)
    # 最优机会
    best = opportunities[0]
    # 记录最优折扣
    logging.info(f"Planning Agent has identified the best deal has discount ${best.discount:.2f}")
    # 超过阈值才推送告警
    if best.discount > DEAL_THRESHOLD:
        # MessagingAgent.alert
        messenger.alert(best)
    # 本轮结束日志
    logging.info("Planning Agent has completed a run")
    # 仅当超过阈值时返回 best，否则 None
    return best if best.discount > DEAL_THRESHOLD else None 


In [ ]:
# ========== 执行一轮规划主流程 ==========
main()
